<a href="https://colab.research.google.com/github/1stzl01/Reserving/blob/main/Christofides_Verrall_Zhi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stochastic Claims Reserving using Log-Linear Regression
### Based on the papers by S. Christofides, R.J. Verrall, and Z. Li

While the traditional chain ladder method is popular, it is purely deterministic and cannot provide statistical confidence intervals. Furthermore, its forecasts can be highly sensitive to random outliers in the data.

This notebook illustrates a stochastic approach: **Log-Incremental Regression**. By taking the natural logarithms of incremental payments, we can frame the claims development process as a formal multiple regression model. This allows us to:
1. Estimate parameters using Ordinary Least Squares (OLS).
2. Use matrix algebra to calculate exact standard errors and variance-covariance matrices for future claims.
3. Handle negative incremental payments (e.g., from salvage or reinsurance recoveries) by optimizing a threshold parameter via Maximum Likelihood Estimation (MLE).

In [3]:
import numpy as np
import pandas as pd
import scipy.optimize as opt

# ==========================================
# 1. LOAD THE DATA (CHRISTOFIDES 4x4 EXAMPLE)
# ==========================================
I = 4 # Size of the triangle

# Incremental payments (using parentheses to prevent formatting errors!)
P = np.array((
    (11073,  6427,  1839,   766),
    (14799,  9357,  2344, np.nan),
    (15636, 10523, np.nan, np.nan),
    (16913, np.nan, np.nan, np.nan)
))

print("Incremental Paid Claims Triangle:")
display(pd.DataFrame(P))

# Extract the valid (non-NaN) data points
valid_indices = np.where(~np.isnan(P))
P_valid = P[valid_indices]

# Transform the data into log-space (Y_ij = ln(P_ij))
Y = np.log(P_valid)
N = len(Y) # Number of observations (Now correctly 10)
print(f"\nNumber of valid observations (N): {N}")

Incremental Paid Claims Triangle:


,0,1,2,3
0,11073.0,6427.0,1839.0,766.0
1,14799.0,9357.0,2344.0,NaN
2,15636.0,10523.0,NaN,NaN
3,16913.0,NaN,NaN,NaN



Number of valid observations (N): 10


### Step 1: Setting up the Design Matrix
The model assumes that the log-incremental payment ($Y_{ij}$) is the sum of an accident year effect ($a_i$) and a development year effect ($b_j$), plus a normal error term ($e_{ij}$):
$Y_{ij} = a_i + b_j + e_{ij}$

To prevent the equations from being singular (having infinite solutions), we set the first development parameter ($b_0$) to zero. This leaves us with 7 parameters to estimate for a 4x4 triangle: $a_0, a_1, a_2, a_3, b_1, b_2, b_3$.

We build a "Design Matrix" (X) populated with 1s and 0s to map each known data point to its corresponding parameters.

In [4]:
# ==========================================
# 2. BUILD THE DESIGN MATRIX (X)
# ==========================================
num_params = I + (I - 1) # 4 accident parameters + 3 development parameters
X = np.zeros((N, num_params))

# Extract row and column arrays separately to avoid bracket formatting issues
row_coords = valid_indices[ 0 ]
col_coords = valid_indices[ 1 ]

# Populate the Design Matrix
for row_idx, (i, j) in enumerate(zip(row_coords, col_coords)):
    # Accident year parameter (a_i)
    X[row_idx, i] = 1
    # Development year parameter (b_j)
    if j > 0:
        X[row_idx, I + j - 1] = 1

# Display the Design Matrix for visual confirmation
param_names = [f"a_{i}" for i in range(I)] + [f"b_{j}" for j in range(1, I)]
X_df = pd.DataFrame(X, columns=param_names, index=[f"Y_{i}{j}" for i, j in zip(row_coords, col_coords)])

print("Design Matrix (X) mapping Y_ij to parameters:")
display(X_df)

Design Matrix (X) mapping Y_ij to parameters:


,a_0,a_1,a_2,a_3,b_1,b_2,b_3
Y_00,1.0,0.0,0.0,0.0,0.0,0.0,0.0
Y_01,1.0,0.0,0.0,0.0,1.0,0.0,0.0
Y_02,1.0,0.0,0.0,0.0,0.0,1.0,0.0
Y_03,1.0,0.0,0.0,0.0,0.0,0.0,1.0
Y_10,0.0,1.0,0.0,0.0,0.0,0.0,0.0
Y_11,0.0,1.0,0.0,0.0,1.0,0.0,0.0
Y_12,0.0,1.0,0.0,0.0,0.0,1.0,0.0
Y_20,0.0,0.0,1.0,0.0,0.0,0.0,0.0
Y_21,0.0,0.0,1.0,0.0,1.0,0.0,0.0
Y_30,0.0,0.0,0.0,1.0,0.0,0.0,0.0


### Step 2: Multiple Regression & Parameter Estimation
With our data in log-space ($Y$) and our design matrix ($X$), we can find the parameter estimates ($\hat{\beta}$) using the closed-form Ordinary Least Squares matrix equation:
$\hat{\beta} = (X^T X)^{-1} X^T Y$

We also calculate the model's underlying variance ($\sigma^2$), which is the sum of the squared residuals divided by the degrees of freedom (Observations - Parameters).


In [5]:
# ==========================================
# 3. ORDINARY LEAST SQUARES (OLS) REGRESSION
# ==========================================

# Calculate parameter estimates: beta = (X.T * X)^-1 * X.T * Y
XTX_inv = np.linalg.inv(X.T @ X)
beta = XTX_inv @ X.T @ Y

# Calculate the residuals and the model variance (sigma^2)
Y_fitted = X @ beta
residuals = Y - Y_fitted
degrees_of_freedom = N - num_params
sigma2 = np.sum(residuals**2) / degrees_of_freedom

print("Parameter Estimates (Beta):")
for name, val in zip(param_names, beta):
    print(f"{name} = {val:.3f}")

print(f"\nModel Variance (sigma^2): {sigma2:.5f}")
print(f"Standard Error of Y Estimate: {np.sqrt(sigma2):.4f}")

Parameter Estimates (Beta):
a_0 = 9.288
a_1 = 9.591
a_2 = 9.692
a_3 = 9.736
b_1 = -0.466
b_2 = -1.801
b_3 = -2.647

Model Variance (sigma^2): 0.00274
Standard Error of Y Estimate: 0.0524


### Step 3: Projecting Future Payments and Standard Errors
To forecast the empty lower-right half of the triangle, we construct a "Future Design Matrix" ($X_f$) representing the missing $i, j$ coordinates.

The predicted log-values are exactly $\hat{Y}_{ij} = X_f \hat{\beta}$.
However, converting these back to actual currencies ($P_{ij}$) requires adjusting for the log-normal distribution's skewness. The expected payment in the original space is:
$P_{ij} = \exp(\hat{Y}_{ij} + 0.5 \cdot \text{TotalVar}(\hat{Y}_{ij}))$

The `TotalVar` includes both the statistical noise ($\sigma^2$) and the estimation error of the parameters (derived from the variance-covariance matrix).

In [6]:
# ==========================================
# 4. FUTURE PROJECTIONS & COVARIANCE MATRIX
# ==========================================

# Identify the missing indices (lower right triangle)
missing_indices = np.where(np.isnan(P))

# Extract row and column arrays separately to avoid bracket formatting issues
missing_rows = missing_indices[ 0 ]
missing_cols = missing_indices[ 1 ]
num_future = len(missing_rows)

# Build Future Design Matrix (X_f)
X_f = np.zeros((num_future, num_params))
future_labels = []

for row_idx, (i, j) in enumerate(zip(missing_rows, missing_cols)):
    future_labels.append(f"P_{i}{j}")
    X_f[row_idx, i] = 1
    if j > 0:
        X_f[row_idx, I + j - 1] = 1

# Calculate predicted future log-values
Y_f = X_f @ beta

# Calculate the Variance-Covariance matrix of the future log-values
# Formula: sigma^2 * [X_f * (X.T * X)^-1 * X_f.T]
Cov_Y_f = sigma2 * (X_f @ XTX_inv @ X_f.T)

# The total variance for each projection is the diagonal of the covariance matrix + model variance
Var_Y_f = np.diag(Cov_Y_f) + sigma2

# Convert log-predictions back to original currency space (accounting for log-normal skew)
P_f = np.exp(Y_f + 0.5 * Var_Y_f)

# Calculate standard error for each individual projected payment
SE_P_f = P_f * np.sqrt(np.exp(Var_Y_f) - 1)

# Calculate overall reserve and its standard error (incorporating all covariances)
Cov_P_f = np.zeros((num_future, num_future))
for row in range(num_future):
    for col in range(num_future):
        if row == col:
            Cov_P_f[row, col] = SE_P_f[row]**2
        else:
            # Covariance in original space formula
            Cov_P_f[row, col] = P_f[row] * P_f[col] * (np.exp(Cov_Y_f[row, col]) - 1)

overall_reserve = np.sum(P_f)
overall_variance = np.sum(Cov_P_f)
overall_se = np.sqrt(overall_variance)

# Output individual predictions
print("Projected Future Incremental Payments:")
results = pd.DataFrame({'Predicted Payment': P_f, 'Std Error': SE_P_f}, index=future_labels)
display(results.round(0))

print(f"\nOverall Estimated Reserve: {overall_reserve:,.0f}")
print(f"Overall Standard Error: {overall_se:,.0f} (CV: {(overall_se/overall_reserve)*100:.1f}%)")

Projected Future Incremental Payments:


,Predicted Payment,Std Error
P_13,1041.0,89.0
P_22,2681.0,211.0
P_23,1152.0,103.0
P_31,10650.0,913.0
P_32,2803.0,251.0
P_33,1204.0,120.0



Overall Estimated Reserve: 19,531
Overall Standard Error: 1,181 (CV: 6.0%)


### Step 4: Extension - Handling Negative Increments
A fundamental problem with log-linear models is that they crash if the data contains negative incremental payments (e.g., from salvage or net-of-reinsurance data), because you cannot take the logarithm of a negative number.

Instead of arbitrarily setting negative values to zero or adding a random constant, R.J. Verrall and Z. Li proved that you can treat this as a 3-parameter log-normal distribution: $Log(P_{ij} + c) = Y_{ij}$.

We can use Python's optimization libraries to automatically search for the "threshold parameter" ($c$) that maximizes the Log-Likelihood of the distribution.

In [7]:
# ==========================================
# 5. MAXIMUM LIKELIHOOD FOR NEGATIVE INCREMENTS
# ==========================================

# Introduce an artificial negative increment safely without changing the matrix shape.
P_neg = P.copy()


# Move the negative increment to the "tail" (e.g., Acc Yr 0, Dev Yr 3)
row_idx = 0
col_idx = 3
P_neg[row_idx, col_idx] = -500



print("New Data with Negative Increment:")
display(pd.DataFrame(P_neg))

def negative_log_likelihood(params):
    # Extract 'c' safely using .item() to avoid bracket formatting bugs
    c = params.item()

    # Extract valid data points
    P_valid_neg = P_neg[~np.isnan(P_neg)]

    # If c is not large enough to make all data positive, it's an invalid guess
    if np.any(P_valid_neg + c <= 0):
        return np.inf

    # Adjust data with c and take logarithms
    Y_adj = np.log(P_valid_neg + c)

    # Given this c, the MLE for the betas is still the OLS solution
    beta_adj = XTX_inv @ X.T @ Y_adj
    residuals_adj = Y_adj - X @ beta_adj

    # For MLE, the variance divides by N (not N - parameters)
    sigma2_mle = np.mean(residuals_adj**2)

    # Calculate the log-likelihood of the normal distribution
    ll = -np.sum(Y_adj) - (N/2)*np.log(2 * np.pi * sigma2_mle) - np.sum(residuals_adj**2)/(2 * sigma2_mle)

    return -ll # Return negative LL because scipy.optimize MINIMIZES functions

# Provide a starting guess for 'c' (using a tuple to avoid bracket formatting)
# 1. Define a strict lower bound that is slightly higher than the absolute minimum (e.g., 500.1)
min_val = abs(np.min(P_neg[~np.isnan(P_neg)]))
strict_lower_bound = min_val + 0.1

# 2. Set bounds for the optimizer: (lower_bound, upper_bound)
# None means there is no upper limit for c
bnds = [(strict_lower_bound, None)]
c_initial_guess = (min_val + 10, )

# 3. Run a Bounded optimization algorithm (L-BFGS-B)
result = opt.minimize(negative_log_likelihood,
                      x0=c_initial_guess,
                      method='L-BFGS-B',
                      bounds=bnds)

optimal_c = result.x.item()
print(f"Optimization Successful: {result.success}")
print(f"Optimal Threshold Parameter (c) = {optimal_c:.2f}")

New Data with Negative Increment:


,0,1,2,3
0,11073.0,6427.0,1839.0,-500.0
1,14799.0,9357.0,2344.0,NaN
2,15636.0,10523.0,NaN,NaN
3,16913.0,NaN,NaN,NaN


Optimization Successful: True
Optimal Threshold Parameter (c) = 500.10
